# Exploratory Data Analysis (EDA) - Credit Card Fraud
> **Note:** Exploratory diagnostic notebook. Production preprocessing (`src/ingestion/utils.py`) deduplicates and performs a stratified train/test split *before* fitting `RobustScaler` (`Amount`) and `StandardScaler` (`Time`) strictly on the training split.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler, StandardScaler

sns.set_theme(style="whitegrid")
DATA_PATH = Path("../data/creditcard.csv")
df = pd.read_csv(DATA_PATH).drop_duplicates().reset_index(drop=True)
print(f"Deduplicated Dataset Shape: {df.shape}")

In [ ]:
class_counts = df["Class"].value_counts()
fraud_rate = (class_counts[1] / len(df)) * 100
print(
    f"Legitimate (0): {class_counts[0]:,} | Fraud (1): {class_counts[1]:,} ({fraud_rate:.3f}%)"
)

plt.figure(figsize=(6, 4))
sns.countplot(
    x="Class", hue="Class", data=df, palette=["#2ecc71", "#e74c3c"], legend=False
)
plt.yscale("log")
plt.title("Class Distribution (Log Scale)")
plt.tight_layout()

In [ ]:
# Fit scalers strictly on training split to prevent train/test leakage
train_df, test_df = train_test_split(
    df, test_size=0.20, random_state=42, stratify=df["Class"]
)
rob_scaler = RobustScaler()
std_scaler = StandardScaler()

train_df = train_df.copy()
test_df = test_df.copy()
train_df["scaled_amount"] = rob_scaler.fit_transform(train_df[["Amount"]])
train_df["scaled_time"] = std_scaler.fit_transform(train_df[["Time"]])
test_df["scaled_amount"] = rob_scaler.transform(test_df[["Amount"]])
test_df["scaled_time"] = std_scaler.transform(test_df[["Time"]])
print("Leakage-free train/test scaling complete:", train_df.shape, test_df.shape)